# Análise Exploratória de Dados — Lending Club
## Plataforma IRB de Risco de Crédito — TribeTech

**Objectivo:** Análise exploratória dos dados Lending Club (2007–2018) para construção dos modelos IRB regulatórios (PD, LGD, EAD) em conformidade com EBA/GL/2017/06.

**Dataset:** 2.2M+ empréstimos pessoais nos EUA  
**Período:** Janeiro 2007 – Dezembro 2018 Q4  
**Variável Target:** `is_default` (conforme definição EBA Artigo 178 CRR)

---

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

# Estilo TribeTech
plt.rcParams.update({
    'figure.facecolor': '#141720',
    'axes.facecolor':   '#1C2030',
    'axes.edgecolor':   '#2E3448',
    'axes.labelcolor':  '#E8EAF0',
    'xtick.color':      '#8B93A8',
    'ytick.color':      '#8B93A8',
    'text.color':       '#E8EAF0',
    'grid.color':       '#2E3448',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'font.family':      'sans-serif',
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
})

TRB_RED    = '#E63C2F'
TRB_ORANGE = '#F97316'
TRB_BLUE   = '#3B82F6'
TRB_GREEN  = '#22C55E'
GRADE_COLORS = ['#22C55E','#3B82F6','#F59E0B','#F97316','#EF4444','#E63C2F','#64748B']

DATA_PATH = '/home/alexmendes/bank_tech/Data/accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv'

print('✓ Bibliotecas carregadas')

## 1. Carregamento Eficiente com DuckDB

O ficheiro CSV tem **1.6 GB** — utilizamos DuckDB para processar directamente em SQL sem carregar tudo para memória.

In [ ]:
con = duckdb.connect()

# Contagem total sem carregar
total = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{DATA_PATH}', ignore_errors=true)").fetchone()[0]
print(f'Total de registos: {total:,}')

# Amostra estratificada para EDA
df = con.execute(f"""
    SELECT
        loan_amnt, funded_amnt, term, int_rate, installment,
        grade, sub_grade, emp_length, home_ownership, annual_inc,
        verification_status, loan_status, purpose, dti,
        delinq_2yrs, fico_range_low, fico_range_high,
        inq_last_6mths, open_acc, pub_rec, revol_bal, revol_util,
        total_acc, recoveries, issue_d
    FROM read_csv_auto('{DATA_PATH}', ignore_errors=true)
    WHERE loan_status IS NOT NULL
      AND annual_inc > 0
      AND grade IS NOT NULL
    USING SAMPLE 200000 ROWS
""").df()

con.close()
print(f'Amostra EDA: {len(df):,} linhas | {df.shape[1]} colunas')

In [ ]:
import sys
sys.path.insert(0, '../fastapi_ml')
from training.feature_engineering import create_default_flag, engineer_features

df = create_default_flag(df)
df = engineer_features(df)

print(f'Defaults: {df["is_default"].sum():,} ({df["is_default"].mean()*100:.1f}%)')
df.head(3)

## 2. Qualidade dos Dados

Verificação de valores em falta, outliers e consistência — requisito EBA para documentação de gestão de dados.

In [ ]:
# Missings por variável
key_cols = ['loan_amnt','int_rate','annual_inc','dti','fico_avg',
            'emp_length','revol_util','open_acc','pub_rec','grade']

missing = pd.DataFrame({
    'missing_n':   df[key_cols].isna().sum(),
    'missing_pct': df[key_cols].isna().mean() * 100,
    'dtype':       df[key_cols].dtypes,
}).sort_values('missing_pct', ascending=False)

print('=== Qualidade de Dados ===')
print(missing.to_string())

## 3. Distribuição da Variável Target (is_default)

Definição conforme EBA Artigo 178 CRR: considera-se incumprimento quando o devedor está em atraso > 90 dias ou quando é provável que não cumpra sem recurso a garantia.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Distribuição do Target — is_default', fontsize=14, fontweight='bold')

# 1. Pie chart
ax = axes[0]
counts = df['is_default'].value_counts()
ax.pie(counts, labels=['Cumpridor', 'Incumpridor'],
       colors=[TRB_GREEN, TRB_RED], autopct='%1.1f%%',
       startangle=90, textprops={'color': '#E8EAF0'})
ax.set_title('Proporção Global')

# 2. Default rate por grade
ax = axes[1]
grade_dr = df.groupby('grade')['is_default'].mean() * 100
bars = ax.bar(grade_dr.index, grade_dr.values,
              color=GRADE_COLORS[:len(grade_dr)], edgecolor='#2E3448', linewidth=0.5)
ax.set_title('Taxa de Default por Grade')
ax.set_ylabel('Taxa de Default (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, grade_dr.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9, color='#E8EAF0')

# 3. Default rate temporal
ax = axes[2]
df['issue_year'] = pd.to_datetime(df['issue_d'], errors='coerce').dt.year
yearly_dr = df.groupby('issue_year')['is_default'].mean() * 100
ax.plot(yearly_dr.index, yearly_dr.values,
        color=TRB_RED, linewidth=2, marker='o', markersize=5)
ax.fill_between(yearly_dr.index, yearly_dr.values, alpha=0.15, color=TRB_RED)
ax.set_title('Evolução da Taxa de Default')
ax.set_ylabel('Taxa de Default (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/eda_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Distribuição das Variáveis Preditoras

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Distribuição das Variáveis Preditoras — Bons vs. Maus', fontsize=14, fontweight='bold')

features_to_plot = [
    ('fico_avg',   'FICO Score'),
    ('dti',        'DTI (%)'),
    ('int_rate',   'Taxa de Juro'),
    ('annual_inc', 'Rendimento Anual'),
    ('loan_amnt',  'Montante Empréstimo'),
    ('revol_util', 'Utilização Revolving (%)'),
]

for (feat, label), ax in zip(features_to_plot, axes.flatten()):
    data_good = df.loc[df['is_default'] == 0, feat].dropna()
    data_bad  = df.loc[df['is_default'] == 1, feat].dropna()

    # Remover outliers extremos para visualização
    p5, p95 = data_good.quantile(0.02), data_good.quantile(0.98)
    data_good = data_good.clip(p5, p95)
    data_bad  = data_bad.clip(p5, p95)

    ax.hist(data_good, bins=40, alpha=0.6, color=TRB_GREEN, label='Cumpridor', density=True)
    ax.hist(data_bad,  bins=40, alpha=0.6, color=TRB_RED,   label='Incumpridor', density=True)
    ax.set_title(label)
    ax.set_ylabel('Densidade')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

    # KS statistic
    from scipy.stats import ks_2samp
    ks, pval = ks_2samp(data_good, data_bad)
    ax.set_xlabel(f'KS = {ks:.3f}', fontsize=9, color=TRB_ORANGE)

plt.tight_layout()
plt.savefig('../data/processed/eda_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Análise de Correlação

In [ ]:
num_cols = ['is_default','fico_avg','dti','int_rate','annual_inc',
            'loan_amnt','revol_util','emp_length','open_acc','pub_rec',
            'loan_to_income','grade_num']

corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))

cmap = sns.diverging_palette(10, 220, s=90, l=40, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, center=0,
            vmin=-1, vmax=1, annot=True, fmt='.2f',
            linewidths=0.5, linecolor='#2E3448',
            annot_kws={'size': 9}, ax=ax)

ax.set_title('Matriz de Correlação — Variáveis IRB', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/eda_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Correlações com target
print('\n=== Correlação com is_default ===')
print(corr['is_default'].drop('is_default').sort_values(key=abs, ascending=False).to_string())

## 6. Análise de Portfólio por Grade

In [ ]:
grade_stats = df.groupby('grade').agg(
    total=('loan_amnt', 'count'),
    exposicao=('loan_amnt', 'sum'),
    ticket_medio=('loan_amnt', 'mean'),
    fico_medio=('fico_avg', 'mean'),
    dti_medio=('dti', 'mean'),
    taxa_juro_media=('int_rate', 'mean'),
    taxa_default=('is_default', 'mean'),
).reset_index()

grade_stats['exposicao_M'] = grade_stats['exposicao'] / 1e6
grade_stats['taxa_default_pct'] = grade_stats['taxa_default'] * 100

print('=== Resumo do Portfólio por Grade ===')
print(grade_stats[['grade','total','exposicao_M','fico_medio','dti_medio',
                    'taxa_juro_media','taxa_default_pct']].to_string(index=False))

# Gráfico
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Análise de Portfólio por Grade', fontsize=13, fontweight='bold')

ax = axes[0]
ax2 = ax.twinx()
bars = ax.bar(grade_stats['grade'], grade_stats['exposicao_M'],
              color=[c+'99' for c in GRADE_COLORS], edgecolor=GRADE_COLORS, linewidth=1.5)
ax2.plot(grade_stats['grade'], grade_stats['taxa_default_pct'],
         color=TRB_RED, linewidth=2.5, marker='D', markersize=7, zorder=5)
ax.set_xlabel('Grade')
ax.set_ylabel('Exposição (M USD)', color='#E8EAF0')
ax2.set_ylabel('Taxa de Default (%)', color=TRB_RED)
ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title('Exposição e Default Rate por Grade')
ax.grid(axis='y', alpha=0.3)

ax = axes[1]
ax.scatter(grade_stats['fico_medio'], grade_stats['taxa_default_pct'],
           s=grade_stats['total']/50, c=GRADE_COLORS[:len(grade_stats)],
           edgecolors='white', linewidth=1.5, zorder=5)
for _, row in grade_stats.iterrows():
    ax.annotate(row['grade'], (row['fico_medio'], row['taxa_default_pct']),
                fontsize=11, fontweight='bold', ha='center', va='bottom',
                xytext=(0, 8), textcoords='offset points')
ax.set_xlabel('FICO Score Médio')
ax.set_ylabel('Taxa de Default (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title('FICO vs. Default Rate (tamanho = volume)')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/eda_portfolio_grade.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Vintage Analysis (Análise de Coorte)

Análise regulatória obrigatória: taxa de default por trimestre de emissão (cohort).

In [ ]:
df['issue_quarter'] = pd.to_datetime(df['issue_d'], errors='coerce').dt.to_period('Q').astype(str)

vintage = df.groupby(['issue_quarter','grade'])['is_default'].mean().unstack('grade') * 100
vintage = vintage.dropna(how='all').tail(20)  # últimos 5 anos

fig, ax = plt.subplots(figsize=(14, 6))
for grade, color in zip(vintage.columns, GRADE_COLORS):
    if grade in vintage.columns:
        ax.plot(range(len(vintage)), vintage[grade].values,
                label=f'Grade {grade}', color=color, linewidth=1.5, marker='o', markersize=3)

ax.set_xticks(range(len(vintage)))
ax.set_xticklabels(vintage.index, rotation=45, ha='right', fontsize=8)
ax.set_title('Vintage Analysis — Taxa de Default por Coorte Trimestral', fontweight='bold')
ax.set_ylabel('Taxa de Default (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend(loc='upper left', fontsize=9, ncol=4)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/eda_vintage_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Estatísticas Descritivas — Resumo Executivo

In [ ]:
print('=' * 60)
print('  SUMÁRIO EXECUTIVO — Análise EDA')
print('=' * 60)

print(f'\n📊 Dimensão da Amostra: {len(df):,} empréstimos')
print(f'   Taxa de Default Global: {df["is_default"].mean()*100:.2f}%')
print(f'   Exposição Total:        ${df["loan_amnt"].sum()/1e6:,.1f}M')
print(f'   Ticket Médio:           ${df["loan_amnt"].mean():,.0f}')
print(f'   FICO Score Médio:       {df["fico_avg"].mean():.0f}')
print(f'   DTI Médio:              {df["dti"].mean():.1f}%')

print('\n🎯 Top 3 Features (Correlação com Default):')
top_corr = df[['is_default','fico_avg','dti','int_rate','grade_num',
               'revol_util','loan_to_income']].corr()['is_default'].drop('is_default')
for feat, corr_val in top_corr.abs().sort_values(ascending=False).head(3).items():
    direction = '↑ risco' if top_corr[feat] > 0 else '↓ risco'
    print(f'   {feat:20s}: r={top_corr[feat]:+.3f} ({direction})')

print('\n📋 Grade com maior risco:',
      df.groupby('grade')['is_default'].mean().idxmax(), 
      f"({df.groupby('grade')['is_default'].mean().max()*100:.1f}% default rate)")

print('\n✅ EDA concluída — pronto para Feature Engineering (WoE/IV)')